# ODI to Databricks Migration

## Source File: `TARGET`

## Converted: 2024-05-15T12:00:00Z

## Description: Direct INSERT from HR.LOCATIONS to HR.TRG_LOC

In [ ]:
dbutils.widgets.text("DATASOURCE_NUM_ID", "")
dbutils.widgets.text("ETL_PROC_WID", "")
dbutils.widgets.text("ODI_SESS_NO", "")

# ETL Parameters

In [ ]:
%sql
SELECT
  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
  '${ETL_PROC_WID}' AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO;

# Merge into Target

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}: Insert into Target Location table
INSERT 
  INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID 
  ) 
SELECT 
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID  
FROM 
  workspace.hr.locations AS LOCATIONS;

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were created in this process to drop.

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS trg_loc_record_count FROM workspace.hr.trg_loc;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: Oracle schema `HR` converted to `workspace.hr`. Table names `TRG_LOC` and `LOCATIONS` converted to lowercase `trg_loc` and `locations` respectively, and prefixed with `workspace.hr`.
2.  **Oracle Hints**: `/*+ APPEND PARALLEL */` hint has been removed as it is Oracle-specific and not applicable to Databricks Spark SQL.
3.  **Data Types**: The `INSERT` statement does not include DDL. Ensure that the target table `workspace.hr.trg_loc` has its schema defined with appropriate Spark SQL data types (e.g., `BIGINT` for IDs, `STRING` for VARCHAR2, `TIMESTAMP` for DATE columns if they were present). You may need to create the `workspace.hr.trg_loc` table DDL manually before running this notebook if it doesn't already exist, ensuring column types are compatible with the source `workspace.hr.locations` table.
4.  **No ODI Parameters**: No explicit ODI `#GLOBAL` or `#SCHEMA` parameters were found in the provided SQL. Standard global parameters (`DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been added as widgets based on the generic example, but are not used in this specific SQL statement.
5.  **No Staging/Flow Tables**: This particular ODI task involves a direct insert from a source table to a target table without intermediate `C$` or `I$` staging/flow tables.